In this notebook, we analyze the scope diversity using the iSim reported by López-Pérez, Kim, and Miranda-Quintana (Digital Discovery, 2024, 3, 1160–1171).

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..','..','..')))
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Code.benchmark import Benchmark
from Code.utils import obtain_full_covar_matrix
from sklearn.preprocessing import MinMaxScaler
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Draw
from rdkit import DataStructs
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import colorsys


# Doyle colors
doyle_colors = ["#CE4C6F", "#1561C2", "#188F9D","#C4ADA2","#515798", "#CB7D85", "#A9A9A9"]
# extension of palette with lighter and darker versions
def adjust_lightness(color, factor=1.2):
    """
    Function to make colors lighter (factor > 1) or darker (factor < 1).
    """
    r, g, b = mcolors.to_rgb(color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0, min(1, l * factor))
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return mcolors.to_hex((r, g, b))

lighter = [adjust_lightness(c, 1.2) for c in doyle_colors]
darker  = [adjust_lightness(c, 0.7) for c in doyle_colors]
all_colors = doyle_colors + darker[::-1] + lighter[::-1] 

# Save the categorical colormap
cat_cmap = ListedColormap(all_colors, name="Doyle_cat")
plt.colormaps.register(cat_cmap)

# Define and save a continuous colormap
colors = [doyle_colors[1],"#FFFFFFD1",doyle_colors[0]]
cont_cmap = LinearSegmentedColormap.from_list("Doyle_cont", colors)
plt.colormaps.register(cont_cmap)
wdir = Path(".")


# General plt parameters
plt.rcParams.update({
    "axes.titlesize": 20,        # Subplot title
    "axes.labelsize": 16,        # X and Y labels
    "figure.titlesize": 24,      # Suptitle
    "xtick.labelsize": 14,       # X tick labels
    "ytick.labelsize": 14,       # Y tick labels
    "legend.fontsize": 14,       # Legend text
    "legend.title_fontsize": 14, # Legend titles
    "font.family": "Helvetica"   # Font
    })

In [2]:
# define a couple things
objectives = ["yield"]
directory = "."
wdir = Path(directory)
datasets = ["high","medium","low"]
dfs_labelled = {dset: pd.read_csv(f"./../Amide_data/Datasets/amide_dset_dft_subs_{dset}-yielding.csv", 
                                  index_col=0,header=0) for dset in datasets}

In [3]:
# NOTE: The functions in this cell were directly taken from the iSim package (https://github.com/mqcomplab/iSIM/tree/main)

def calculate_isim(data, n_objects = None, n_ary = 'RR'):
    """Calculate the iSIM index for RR, JT, or SM

    Arguments
    ---------
    data : np.ndarray
        Array of arrays, each sub-array contains the binary object 
        OR Array with the columnwise sum, if so specify n_objects
    
    n_objects : int
        Number of objects, only necessary if the column wize sum is the input data.

    n_ary : str
        String with the initials of the desired similarity index to calculate the iSIM from. 
        Only RR, JT, or SM are available. For other indexes use gen_sim_dict.

    Returns
    -------
    isim : float
        iSIM index for the specified similarity index.
    """

    # Check if the data is a np.ndarray of a list
    if not isinstance(data, np.ndarray):
        raise TypeError("Warning: Input data is not a np.ndarray, to secure the right results please input the right data type")
    
    if data.ndim == 1:
        c_total = data
        if not n_objects:
            raise ValueError("Input data is the columnwise sum, please specify number of objects")
    else:
        c_total = np.sum(data, axis = 0)
        if not n_objects:
            n_objects = len(data)      
        elif n_objects and n_objects != len(data):
            print("Warning, specified number of objects is different from the number of objects in data")
            n_objects = len(data)
            print("Doing calculations with", n_objects, "objects.")

    # Calculate only necessary counters for the desired index 

    if n_ary == 'RR':
        a = np.sum(c_total * (c_total - 1) / 2)
        p = n_objects * (n_objects - 1) * len(c_total) / 2

        return a/p
    
    elif n_ary == 'JT':
        a = np.sum(c_total * (c_total - 1) / 2)
        off_coincidences = n_objects - c_total
        total_dis = np.sum(off_coincidences * c_total)

        return a/(a + total_dis)
    
    elif n_ary == 'SM':
        a = np.sum(c_total * (c_total - 1) / 2)
        off_coincidences = n_objects - c_total
        d = np.sum(off_coincidences * (off_coincidences - 1) / 2)
        p = n_objects * (n_objects - 1) * len(c_total) / 2

        return (a + d)/p


def binary_fps(smiles: list, fp_type: str = 'RDKIT', n_bits: int = 2048):
    """
    This function generates binary fingerprints for the dataset.
    
    Parameters:
    smiles: list of SMILES strings
    fp_type: type of fingerprint to generate ['RDKIT', 'ECFP4', 'ECFP6', or 'MACCS']
    n_bits: number of bits for the fingerprint
    
    Returns:
    fingerprints: numpy array of fingerprints
    """
    # Generate the fingerprints
    if fp_type == 'RDKIT':
       def generate_fp(mol, fp):
            DataStructs.cDataStructs.ConvertToNumpyArray(Chem.RDKFingerprint(mol), fp)
    elif fp_type == 'ECFP4':
        def generate_fp(mol, fp):
            DataStructs.cDataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits), fp)
    elif fp_type == 'ECFP6':
        def generate_fp(mol, fp):
            DataStructs.cDataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=n_bits), fp)
    elif fp_type == 'MACCS':
        def generate_fp(mol, fp):
            DataStructs.cDataStructs.ConvertToNumpyArray(Chem.rdMolDescriptors.GetMACCSKeysFingerprint(mol), fp)
    else:
        print('Invalid fingerprint type: ', fp_type)
        exit(0)

    fingerprints = []
    for smi in smiles:
        # Generate the mol object
        try:
          mol = Chem.MolFromSmiles(smi)
        except:
          print('Invalid SMILES: ', smi)
          exit(0)

        # Generate the fingerprint and append to the list
        fingerprint = np.array([])
        generate_fp(mol, fingerprint)
        fingerprints.append(fingerprint)
    
    fingerprints = np.array(fingerprints)

    return fingerprints

In [4]:
def iSim_calculation(filename):
    """
    Calculate and return the iSim values for a Benchmark().collect_data() output file.
    Settings: Russel-Rao algorithm; ECFP4 fingerprint
    """

    # read in the raw data file
    df_raw = pd.read_csv(filename, index_col = 0, header = 0)
    # convert the format of the evaluated samples from one string to a list of strings
    df_raw["eval_samples"] = df_raw["eval_samples"].apply(lambda x: [y.strip("'") for y in x[1:-1].split(', ')])

    # loop through the rounds of the run
    smiles_list = []
    for run_round in df_raw.index:
        # get the evaluated samples
        samples = df_raw.loc[run_round,"eval_samples"]
        # record the evaluated samples in the df of the new featurization
        for sample in samples:
            sample = str(sample.encode().decode('unicode_escape'))
            smiles_list.append(sample)
    
    # get fingerprints and calculate iSim
    fps = binary_fps(smiles_list, fp_type="ECFP4", n_bits = 2048)
    isim = calculate_isim(fps, n_ary="RR")

    return isim

### Recalculate the diversity scores with iSim

In [7]:
# calculate the iSim values
isim_results_dict = {}

for dset in datasets:
    isim_results = pd.DataFrame(np.nan, index=["Results"], columns=[])
    for acq in ["EI","Random","Greedy","Conv. selection","Explorative"]:
        if acq == "Random":
            acq_label = "random-selection"
        elif acq == "Conv. selection":
            acq_label = "human-like-acq"
        else:
            acq_label = acq.lower()
        for pruning in [True,False]:
            if pruning:
                pruning_label = "_with-pruning"
                pruning_flag = "with"
            else:
                pruning_label = "_no-pruning"
                pruning_flag = "without"
            if acq == "Conv. selection":
                pruning_label = ""
            isim_vals = []
            folder_path = f"./Results_Data/{dset}-dataset/{acq_label}{pruning_label}/raw_data"
            for filename in os.listdir(folder_path):
                isim = iSim_calculation(f"{folder_path}/{filename}")
                isim_vals.append(isim)
            isim_results.loc["Results", acq+pruning_label] = np.mean(isim_vals)
    isim_results.rename(columns={"EI_pruning":"ScopeBO"},inplace=True)
    label_dict = {col: col for col in isim_results.columns}
    for key,val in label_dict.items():
        if "_no-pruning" in val:
            label_dict[key] = val.split("_")[0]
        elif "pruning" in val:
            label_dict[key] = val.split("_")[0] + " (pruned)"
    isim_results.rename(columns=label_dict,inplace=True)
    isim_results.sort_values(by="Results",inplace=True, axis=1)
    isim_results_dict[dset] = isim_results

In [14]:
for dset in datasets:
    print(f"iSim similarity values for {dset} dataset (lower value indicates lower similarity):")
    results_ranked = isim_results_dict[dset].rank(axis=1, method='min', ascending=True).astype(int)
    isim_results_combined = pd.concat([isim_results_dict[dset], results_ranked], axis=0)
    isim_results_combined.index = ["iSim values","Rankings"]
    isim_results_combined = isim_results_combined.rename(columns={"EI (pruned)": "ScopeBO"})
    order = [
        "Greedy",
        "Greedy (pruned)",
        "EI",
        "ScopeBO",
        "Random",
        "Random (pruned)",
        "Explorative",
        "Explorative (pruned)",
        "Conv. selection",
    ]

    isim_results_combined = isim_results_combined.T.reindex(order)
    display(isim_results_combined)

iSim similarity values for high dataset (lower value indicates lower similarity):


,iSim values,Rankings
Greedy,0.008544,9.0
Greedy (pruned),0.006673,6.0
EI,0.006480,5.0
ScopeBO,0.006273,4.0
Random,0.006882,7.0
Random (pruned),0.006248,3.0
Explorative,0.005869,2.0
Explorative (pruned),0.005852,1.0
Conv. selection,0.007287,8.0


iSim similarity values for medium dataset (lower value indicates lower similarity):


,iSim values,Rankings
Greedy,0.008301,9.0
Greedy (pruned),0.006525,6.0
EI,0.006194,4.0
ScopeBO,0.006061,3.0
Random,0.006882,7.0
Random (pruned),0.006248,5.0
Explorative,0.005804,2.0
Explorative (pruned),0.005801,1.0
Conv. selection,0.007020,8.0


iSim similarity values for low dataset (lower value indicates lower similarity):


,iSim values,Rankings
Greedy,0.008350,9.0
Greedy (pruned),0.006781,6.0
EI,0.006448,5.0
ScopeBO,0.006186,3.0
Random,0.006882,7.0
Random (pruned),0.006248,4.0
Explorative,0.005790,2.0
Explorative (pruned),0.005781,1.0
Conv. selection,0.007174,8.0
